In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import rasterio
import wandb
from sklearn.model_selection import train_test_split

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [3]:
wandb.init(
    project="oil-spill-detection",
    name="CNN-6Conv-32Filters-512x512",
    config={
        "input_shape": (2, 512, 512),
        "conv_layers": 6,
        "filters": 32,
        "kernel_size": 3,
        "pool_size": 2,
        "dense_units": [20, 20],
        "optimizer": "Adam",
        "loss": "BCELoss",
        "epochs": 50,
        "batch_size": 4
    }
)

wandb: Currently logged in as: paras-ningune01 (paras-ningune01-pvg-coet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
IMG_SIZE = (512, 512)

def load_sar_tiff(path):
    with rasterio.open(path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

    vv = np.clip(vv, -35, 5)
    vh = np.clip(vh, -40, 0)

    vv = (vv + 35) / 40
    vh = (vh + 40) / 40

    return np.stack([vv, vh], axis=0)  # (2, H, W)

In [5]:
class OilSpillDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths = paths
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = torch.tensor(load_sar_tiff(self.paths[idx]), dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.augment:
            if random.random() > 0.5:
                image = torch.flip(image, [2])
            if random.random() > 0.5:
                image = torch.flip(image, [1])

            k = random.randint(0, 3)
            image = torch.rot90(image, k, [1, 2])

            tx = int(random.uniform(-0.05, 0.05) * IMG_SIZE[0])
            ty = int(random.uniform(-0.05, 0.05) * IMG_SIZE[1])
            image = torch.roll(image, shifts=(tx, ty), dims=(1, 2))

        return image, label

In [6]:
def build_balanced_dataset(root, target_per_class=1200):
    oil = [os.path.join(root, "Oil", f) for f in os.listdir(os.path.join(root, "Oil")) if f.endswith(".tif")]
    no_oil = [os.path.join(root, "No_Oil", f) for f in os.listdir(os.path.join(root, "No_Oil")) if f.endswith(".tif")]
    look = [os.path.join(root, "Lookalike", f) for f in os.listdir(os.path.join(root, "Lookalike")) if f.endswith(".tif")]

    combined_no_oil = no_oil + look
    random.shuffle(combined_no_oil)

    oil = oil[:target_per_class]
    combined_no_oil = combined_no_oil[:target_per_class]

    paths = oil + combined_no_oil
    labels = [1]*len(oil) + [0]*len(combined_no_oil)

    return np.array(paths), np.array(labels)

In [9]:
BASE_PATH = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "Train", "Images")
TEST_IMG_DIR  = os.path.join(BASE_PATH, "Test", "Images")

all_paths, all_labels = build_balanced_dataset(TRAIN_IMG_DIR)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, train_size=0.66, stratify=all_labels, random_state=SEED
)

# Model-1

In [22]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        layers = []
        in_channels = 2

        for _ in range(6):
            layers += [
                nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            ]
            in_channels = 32

        self.conv = nn.Sequential(*layers)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 20),
            nn.ReLU(),
            nn.Linear(20, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

# Model-2

In [11]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        layers = []
        in_channels = 2

        for _ in range(6):
            layers += [
                nn.Conv2d(in_channels, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            ]
            in_channels = 32

        self.conv = nn.Sequential(*layers)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 20),
            nn.ReLU(),
            nn.Linear(20, 1)  # ❗ No sigmoid (see loss)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

In [23]:
def save_checkpoint(model, optimizer, epoch, best_val_loss, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_loss": best_val_loss
    }, path)
    print(f"Checkpoint saved (epoch {epoch})")

In [24]:
def load_checkpoint(model, optimizer, path, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    print(f"Resumed from epoch {checkpoint['epoch'] + 1}")
    return checkpoint["epoch"] + 1, checkpoint["best_val_loss"]

In [25]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [26]:
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device).unsqueeze(1)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item()

            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == y).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [27]:
BASE_PATH = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "Train", "Images")

paths, labels = build_balanced_dataset(TRAIN_IMG_DIR)

train_p, val_p, train_l, val_l = train_test_split(
    paths, labels, train_size=0.66, stratify=labels, random_state=SEED
)

train_loader = DataLoader(
    OilSpillDataset(train_p, train_l, augment=True),
    batch_size=4, shuffle=True, num_workers=0
)

val_loader = DataLoader(
    OilSpillDataset(val_p, val_l, augment=False),
    batch_size=4, shuffle=False, num_workers=0
)

In [28]:
model = CNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

CHECKPOINT_PATH = "checkpoints/cnn_checkpoint.pth"
TOTAL_EPOCHS = 5
EPOCHS_PER_RUN = 5

start_epoch = 0
best_val_loss = float("inf")

if os.path.exists(CHECKPOINT_PATH):
    start_epoch, best_val_loss = load_checkpoint(
        model, optimizer, CHECKPOINT_PATH, device
    )
else:
    start_epoch = 0
    best_val_loss = float("inf")

In [29]:
for run_start in range(start_epoch, TOTAL_EPOCHS, EPOCHS_PER_RUN):

    run_end = min(run_start + EPOCHS_PER_RUN, TOTAL_EPOCHS)
    print(f"\nTraining epochs {run_start} → {run_end - 1}")

    for epoch in range(run_start, run_end):

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion
        )

        val_loss, val_acc = validate(
            model, val_loader, criterion
        )

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        # Save LAST checkpoint every epoch (for safe resume)
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_loss": best_val_loss
        }, CHECKPOINT_LAST)

        # Save BEST checkpoint (for deployment)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_loss": best_val_loss
            }, CHECKPOINT_BEST)

    print("Safe to stop now — resume anytime")


Training epochs 0 → 4


RuntimeError: MPS backend out of memory (MPS allocated: 9.02 GiB, other allocations: 2.72 MiB, max allowed: 9.07 GiB). Tried to allocate 2.00 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).